# 使用三个接收机进行校准

## 简介

人们早已知道，只要使用具有三个接收器且不含内部同轴开关的 VNA，就可以实现完全的误差校正。然而，由于没有现代 VNA 采用这种架构，因此用于进行完全校正测量的软件在当今的现代 VNA 上也无法使用。最近，仅包含三个接收器的频率扩展器的应用变得越来越普遍。此外，低成本的 VNA 正在成为学习射频电子技术的有用工具。一个典型的例子是 NanoVNA，它采用了一种具有三个接收器的设计。因此，对于具有三个接收器且不含内部同轴开关的系统，需要具备完全的误差校正功能。本文档描述了如何使用 [scikit-rf](http://www.scikit-rf.org) 对此类系统进行的双端口测量进行完全校正。

## 理论

下面展示了一个无开关三接收器系统的电路模型。

In [ ]:
from IPython.display import SVG, Image

SVG('three_receiver_cal/pics/vnaBlockDiagramForwardRotated.svg')

为了完全校正任意二端口器件，必须以两种方向测量该器件，分别称为正向和反向。由于没有开关，因此需要操作员手动翻转器件，并保存这两组测量数据。在晶圆级测试中，可以制造两个完全相同的器件，每个器件分别以一种方向放置。无论哪种情况，在进行校正之前，都需要对每个被测器件（DUT）进行两组测量。在实际操作中，虽然器件被翻转了，但我们可以想象器件是静止的，整个 VNA 电路被翻转。这种解释有利于实现，因为可以通过简单地将正向误差系数复制到相应的反向误差系数中，从而重用现有的 12 项校正。`scikit-rf` 在内部就是这样做的。

## 示例演示

本示例演示了如何从在 Agilent PNAX 上使用一组 VDI WR-12 TXRX-RX 频率扩展器头采集的测量数据中创建 [TwoPortOnePath](../../api/calibration/generated/skrf.calibration.calibration.TwoPortOnePath.rst) 和 [EnhancedResponse](../../api/calibration/generated/skrf.calibration.calibration.EnhancedResponse.rst) 校准。通过校正一个不对称的 DUT，可以对这两种算法进行比较。

### 测量校准标准和待测设备 (DUT)首先，使用 VNA 按照标准的 SOLT 校准程序测量开路、短路和负载标准元件的散射参数。我们假设读者已经熟悉 scikit-rf 中的此操作。如果不是，请参阅 [SOLT 示例](./SOLT.ipynb)，其中对此进行了完整描述。在校准具有三个接收器的 VNA 时，过程几乎相同。但请注意，存在一些小的差异。首先，开路、短路和负载标准元件仅在端口 1 处测量，而不是端口 2。VNA 无法以相反的方向进行测量，因此没有理由在端口 2 上执行 OSL 校准（除非需要可选的隔离校准）。尽管如此，scikit-rf 仍然期望将双端口网络作为输入，因此我们仍然将所有结果都保存为双端口网络。在这种情况下，只有 $S_{11}$ 和 $S_{21}$ 是实际测量值，$S_{12}$ 和 $S_{22}$ 未使用，没有意义，并且可能包含任意数据。可以使用全零值作为占位符。在内部，scikit-rf 会将 $S_{22} = S_{11}$ 和 $S_{12} = S_{21}$ 用于计算。然后，以正向方式测量 DUT 作为双端口网络，物理翻转，然后再次以反向方式测量。

### 读取数据

校准标准和待测设备的测量数据是从VNA下载的，具体方法是将原始散射参数数据保存为Touchstone文件存储到磁盘中。

In [ ]:
! ls three_receiver_cal/data/

可以使用以下方法将这些文件读取到 scikit-rf 中，并将其转换为 `Network` 对象。

In [ ]:
import matplotlib.pyplot as plt

import skrf as rf

%matplotlib inline

rf.stylely()


raw = rf.io.general.read_all_networks('three_receiver_cal/data/')
# list the raw measurements
raw.keys()

可以通过字典 `raw` 访问每个 `Network` 对象。

In [ ]:
thru = raw['thru']
thru

如果我们查看“直通”标准的原始测量数据，可以看出只有 $S_{11}$ 和 $S_{21}$ 包含有意义的数据。其他散射参数都是噪声。

In [ ]:
thru.plot_s_db()

### 创建校准

在下面的代码中，通过使用校准标准的相应测量响应和理想响应，创建了一个两端口单路径校准。从磁盘读取测量的网络数据，同时使用 scikit-rf 生成相应的理想响应。关于如何使用 scikit-rf 进行离线校准的更多信息，请参阅[此处](../../tutorials/Calibration.ipynb)。

In [ ]:

from skrf.calibration import TwoPortOnePath
from skrf.constants import mil
from skrf.media import RectangularWaveguide
from skrf.network import two_port_reflect as tpr

# pull frequency information from measurements
frequency = raw['short'].frequency

# the media object
wg = RectangularWaveguide(frequency=frequency, a=120*mil, z0_override=50)

# list of 'ideal' responses of the calibration standards
ideals = [wg.short(nports=2),
          tpr(wg.delay_short( 90,'deg'), wg.match()),
          wg.match(nports=2),
          wg.thru()]

# corresponding measurements to the 'ideals'
measured = [raw['short'],
            raw['quarter wave delay short'],
            raw['load'],
            raw['thru']]

# the Calibration object
cal = TwoPortOnePath(measured = measured, ideals = ideals)

默认情况下，`TwoPortOnePath` 假定在所有校准和测量中，端口 1 都是活动源（因此，只有 $S_{11}$ 和 $S_{21}$ 是有效数据）。如果由于某种原因，有效测量结果显示为 $S_{12}$ 和 $S_{22}$，则在创建 `TwoPortOnePath` 对象时，可以设置可选参数 `source_port=2`。### 应用校正

使用 3 接收器系统可以进行两种类型的校正。1. 完全校正 (TwoPortOnePath)2. 部分校正 (EnhancedResponse)`scikit-rf` 对这两种校正都使用相同的 `Calibration` 对象，但根据被测设备 (DUT) 的 `type`，采用不同的校正算法。在本示例中使用的 DUT 是一个 WR-15 垫片，与一个 WR-12 1 英寸直波导级联，如图所示。对该 DUT 进行测量时，分别使用*完全*校正和*部分*校正，并在下文中比较结果。

In [ ]:
Image('three_receiver_cal/pics/asymmetic DUT.jpg', width='75%')

#### 完全校正（两端口单路径）

通过在两个方向（**正向**和**反向**）测量每个器件，可以实现完全的校正。为了明确起见，这意味着必须物理地移除DUT，翻转，然后重新插入。然后，将得到的成对的测量值作为元组传递给 `apply_cal()` 函数。该函数会返回一个单独的校正后的响应。

#### 部分校正（增强响应）

如果将单个测量值传递给 `apply_cal()` 函数，则校准将采用部分校正。这种类型的校正被称为“增强响应”校正。根据具体的测量应用，这种类型的校正可能“足够好”，甚至可能是唯一的选择。

### 比较

下面是对上述DUT进行*完整*和*部分*算法校正后的直接比较。结果表明，部分校准会在反射测量中产生较大的波动，并且在传输测量中产生稍大的波动。

In [ ]:
simulation = raw['simulation']

dutf = raw['wr15 shim and swg (forward)']
dutr = raw['wr15 shim and swg (reverse)']

corrected_full =     cal.apply_cal((dutf, dutr))
corrected_partial =  cal.apply_cal(dutf)



# plot results

f, ax = plt.subplots(1,2, figsize=(8,4))

ax[0].set_title ('$S_{11}$')
ax[1].set_title ('$S_{21}$')

corrected_partial.plot_s_db(0,0, label='Partial Correction',ax=ax[0])
corrected_partial.plot_s_db(1,0, label='Partial Correction',ax=ax[1])

corrected_full.plot_s_db(0,0, label='Full Correction', ax = ax[0])
corrected_full.plot_s_db(1,0, label='Full Correction', ax = ax[1])

simulation.plot_s_db(0,0,label='Simulation', ax=ax[0], color='k')
simulation.plot_s_db(1,0,label='Simulation', ax=ax[1], color='k')

plt.tight_layout()

### 如果我的待测器件是对称的怎么办？

如果已知被测设备（DUT）是**互易的**（$S_{21}=S_{12}$）并且是**对称的**（$S_{11}=S_{22}$），那么其正向和反向的响应应该相同。在这种情况下，两次测量设备是没有必要的，可以避免。这在以下示例中进行了说明：[TwoPortOnePath, EnhancedResponse, and FakeFlip](TwoPortOnePath, EnhancedResponse, and FakeFlip.ipynb)